# A numerical study of the time-dependent Schrödinger equation

This notebook solves the time-dependent Schrödinger equation for a quantum wave
packet in one dimension, and uses the solution to illustrate three standard
results of introductory quantum mechanics: the spreading of a free packet,
reflection at a potential, and tunnelling through a barrier.

It was written as a course project, with the aim of making the connection
between the equation and its numerical solution explicit, and is shared so that
it may be read and reused by others studying the same material. The numerical
methods are implemented in the accompanying module `schrodinger.py`; this
notebook concentrates on the physics and on the interpretation of the results.

## 1. The equation

A non-relativistic particle in one dimension is described by a complex wave
function $\psi(x, t)$ obeying the time-dependent Schrödinger equation. With the
convenient choice of units $\hbar = 1$ and $m = \tfrac12$,

$$ i\,\frac{\partial \psi}{\partial t} = -\frac{\partial^2 \psi}{\partial x^2} + V(x)\,\psi . $$

The quantity $|\psi(x, t)|^2$ is the probability density for the particle's
position, so that $\int |\psi|^2\,dx = 1$ at all times. The particle is confined
to a box $[0, L]$ with hard walls, $\psi(0, t) = \psi(L, t) = 0$.

The initial state is a Gaussian wave packet, the quantum state that most closely
resembles a classical particle with a fairly well-defined position and momentum:

$$ \psi(x, 0) = e^{i k_0 x}\, e^{-(x - x_0)^2 / 2\sigma_0^2} . $$

The plane-wave factor $e^{i k_0 x}$ gives the packet a mean momentum $k_0$; with
$m = \tfrac12$ this corresponds to a group velocity $v = 2 k_0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import schrodinger as sch

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Colours used consistently throughout: density, real part, imaginary part, wall.
DENSITY = "#2a9d8f"
REAL = "#e76f51"
IMAG = "#4a6fa5"
WALL = "#e9c46a"

In [ ]:
# The spatial grid and the initial packet.
L = 1.0
k0 = 100.0
n_points = 256

x = np.linspace(0, L, n_points)
dx = x[1] - x[0]

psi0 = sch.gaussian_packet(x, x0=0.25 * L, sigma=0.05 * L, k0=k0)

wavelength = 2 * np.pi / k0
print(f"{n_points} grid points, dx = {dx:.2e}")
print(f"de Broglie wavelength = {wavelength:.3f}, i.e. {wavelength / dx:.0f} points per wavelength")
print(f"total probability at t = 0: {sch.norm(psi0, dx):.6f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(x, np.abs(psi0) ** 2, color=DENSITY, lw=2, label="|ψ|²")
plt.plot(x, psi0.real, color=REAL, lw=0.8, alpha=0.7, label="Re ψ")
plt.xlabel("x")
plt.title("The initial Gaussian packet")
plt.legend()
plt.show()

## 2. Discretisation

The wave function is sampled on the grid, $\psi_j^n \equiv \psi(x_j, t_n)$ with
$x_j = j\,dx$ and $t_n = n\,dt$, and the second derivative is approximated by the
standard three-point formula

$$ \frac{\partial^2 \psi}{\partial x^2}\bigg|_j \approx \frac{\psi_{j+1} - 2\psi_j + \psi_{j-1}}{dx^2} . $$

Two schemes are used to advance the solution in time.

**Explicit scheme.** A forward difference in time, with the right-hand side
evaluated at the current step, gives

$$ \psi_j^{\,n+1} = \psi_j^{\,n} + i\,\frac{dt}{dx^2}\big(\psi_{j+1}^n - 2\psi_j^n + \psi_{j-1}^n\big) - i\,dt\,V_j\,\psi_j^n . $$

This update is immediate and requires no linear algebra. For the Schrödinger
equation, however, it is unstable: the total probability grows with time rather
than remaining constant.

**Crank-Nicolson scheme.** The Hamiltonian is applied half at the old time and
half at the new one,

$$ \Big(1 + \tfrac{i\,dt}{2}\hat H\Big)\psi^{\,n+1} = \Big(1 - \tfrac{i\,dt}{2}\hat H\Big)\psi^{\,n}, \qquad (\hat H\psi)_j = -\frac{\psi_{j+1} - 2\psi_j + \psi_{j-1}}{dx^2} + V_j\psi_j . $$

Each step then requires the solution of a tridiagonal linear system, but the
scheme is stable for any time step and conserves probability. Both schemes are
implemented in `schrodinger.py`.

## 3. The free particle

With $V = 0$ the packet is expected to travel to the right at constant speed and
to spread, since its momentum components propagate at different velocities.

In [ ]:
dt = 1e-6
n_steps = 3000

psi = sch.evolve_crank_nicolson(psi0, sch.free(x), dx, dt, n_steps)

In [ ]:
def show(state, V, title):
    # Plot one snapshot: filled probability density, with real and imaginary parts.
    density = np.abs(state) ** 2
    density = density / density.max()
    scale = max(np.abs(state.real).max(), np.abs(state.imag).max(), 1e-30)

    if V.max() > 0:
        plt.fill_between(x, 0, V / V.max(), color=WALL, alpha=0.5, lw=0, label="V(x)")
    plt.plot(x, state.real / scale, color=REAL, lw=0.8, alpha=0.7, label="Re ψ")
    plt.plot(x, state.imag / scale, color=IMAG, lw=0.8, alpha=0.7, label="Im ψ")
    plt.fill_between(x, 0, density, color=DENSITY, alpha=0.25, lw=0)
    plt.plot(x, density, color=DENSITY, lw=2, label="|ψ|²")

    plt.xlabel("x")
    plt.ylim(-1.15, 1.15)
    plt.title(title)
    plt.legend(fontsize=8)


times_to_show = [0, n_steps // 3, 2 * n_steps // 3]

plt.figure(figsize=(12, 3.2))
for panel, n in enumerate(times_to_show):
    plt.subplot(1, 3, panel + 1)
    show(psi[n], sch.free(x), f"t = {n * dt:.1e}")
plt.tight_layout()
plt.show()

A single figure can capture the entire evolution. Stacking every snapshot of
$|\psi(x, t)|^2$, with time increasing upward, produces a space-time map in which
the free packet appears as a slanted band: slanted because it moves at constant
speed, and widening because it spreads.

In [ ]:
def spacetime(history, title, barrier_position=None):
    # Display |psi(x, t)|^2 as a heat map, with time increasing upward.
    density = np.abs(history) ** 2
    sampled_rows = np.linspace(0, len(history) - 1, 400).astype(int)

    plt.figure(figsize=(6, 4.4))
    plt.imshow(
        density[sampled_rows],
        origin="lower",
        aspect="auto",
        cmap="magma",
        extent=[0, L, 0, len(history) * dt],
    )
    if barrier_position is not None:
        plt.axvline(barrier_position, color="white", lw=1, ls="--", alpha=0.7)

    plt.xlabel("position x")
    plt.ylabel("time t")
    plt.title(title)
    plt.grid(False)
    plt.colorbar(label="|ψ(x, t)|²")
    plt.show()


spacetime(psi, "Free particle: a spreading, drifting packet")

### Conservation of probability

The clearest test of the solver is whether the total probability
$\sum_j |\psi_j|^2\,dx$ remains equal to one. Comparing the two schemes on the
same small time step, the Crank-Nicolson result stays at one to within rounding
error (of order $10^{-13}$), whereas the explicit scheme drifts steadily upward.

In [ ]:
dt_fine = dx ** 2 / 50
n_steps_fine = 2500

cn = sch.evolve_crank_nicolson(psi0, sch.free(x), dx, dt_fine, n_steps_fine)
ex = sch.evolve_explicit(psi0, sch.free(x), dx, dt_fine, n_steps_fine)
t = np.arange(n_steps_fine + 1) * dt_fine

plt.figure(figsize=(7, 4))
plt.semilogy(t, np.abs(sch.norm(ex, dx) - 1), color=REAL, label="explicit")
plt.semilogy(t, np.abs(sch.norm(cn, dx) - 1), color=DENSITY, label="Crank-Nicolson")
plt.xlabel("time t")
plt.ylabel("|total probability − 1|")
plt.title("Conservation of probability for the two schemes")
plt.legend()
plt.show()

print(f"maximum drift, Crank-Nicolson: {np.abs(sch.norm(cn, dx) - 1).max():.1e}")
print(f"maximum drift, explicit:       {np.abs(sch.norm(ex, dx) - 1).max():.1e}")

Increasing the time step towards $dx^2$ makes the explicit error grow more
rapidly and eventually diverge, while the Crank-Nicolson result is essentially
unchanged. This stability is the reason the implicit scheme is used for the
remaining simulations.

## 4. The potential step

The packet now meets a step of height $V_0$ equal to its mean energy
$E = k_0^2$. A classical particle with $E = V_0$ would cross the step; the
quantum packet is instead largely reflected, with an evanescent tail penetrating
a short distance before decaying.

In [ ]:
V0 = k0 ** 2
V_step = sch.step(x, height=V0, position=0.5)
psi = sch.evolve_crank_nicolson(psi0, V_step, dx, dt, n_steps)

plt.figure(figsize=(12, 3.2))
for panel, n in enumerate(times_to_show):
    plt.subplot(1, 3, panel + 1)
    show(psi[n], V_step, f"t = {n * dt:.1e}")
plt.tight_layout()
plt.show()

reflected, transmitted = sch.reflection_transmission(psi[-1], x, split=0.5)
print(f"reflected R = {reflected:.3f}, transmitted T = {transmitted:.3f}, R + T = {reflected + transmitted:.5f}")

## 5. The barrier and quantum tunnelling

A barrier has a finite width, so part of the wave can pass through it and
re-emerge on the far side even when the barrier is as high as the particle's
energy. This is forbidden for a classical particle, but occurs with finite
probability in quantum mechanics.

In [ ]:
V_barrier = sch.barrier(x, height=V0, width=0.03, center=0.5)
psi = sch.evolve_crank_nicolson(psi0, V_barrier, dx, dt, n_steps)

plt.figure(figsize=(12, 3.2))
for panel, n in enumerate(times_to_show):
    plt.subplot(1, 3, panel + 1)
    show(psi[n], V_barrier, f"t = {n * dt:.1e}")
plt.tight_layout()
plt.show()

spacetime(psi, "Tunnelling: a reflected part and a transmitted part", barrier_position=0.5)

The reflected and transmitted fractions settle to definite values whose sum
remains equal to one at every instant, confirming that probability is conserved.

In [ ]:
reflected, transmitted = sch.reflection_transmission(psi, x, split=0.5)
t = np.arange(n_steps + 1) * dt

plt.figure(figsize=(7, 4))
plt.plot(t, reflected, color=REAL, label="R (reflected)")
plt.plot(t, transmitted, color=DENSITY, label="T (transmitted)")
plt.plot(t, reflected + transmitted, "--", color="#444", lw=1, label="R + T")
plt.xlabel("time t")
plt.ylabel("probability")
plt.ylim(-0.05, 1.1)
plt.title("Reflection and transmission over time")
plt.legend()
plt.show()

print(f"final transmission T = {transmitted[-1]:.3f}")

### Dependence on the barrier width

When the barrier is higher than the particle's energy ($E < V_0$), the wave
inside it decays as $e^{-\kappa x}$ with $\kappa = \sqrt{V_0 - E}$, so the
transmitted probability falls off exponentially with the width $l$:

$$ T \propto e^{-2\kappa l} . $$

This is tested below at $E / V_0 = 0.5$ for a range of widths. A slightly wider
packet (narrower in momentum) is used so that the single-energy law is clearer;
the small departure from the ideal slope reflects the packet's residual spread
in energy.

In [ ]:
V_high = 2 * V0
kappa = np.sqrt(V_high - k0 ** 2)
widths = np.linspace(0.01, 0.05, 8)

transmissions = []
for width in widths:
    V_w = sch.barrier(x, V_high, width, 0.5)
    packet = sch.gaussian_packet(x, 0.25 * L, 0.09 * L, k0)
    psi_w = sch.evolve_crank_nicolson(packet, V_w, dx, dt, n_steps)
    transmissions.append(sch.reflection_transmission(psi_w[-1], x, 0.5 + width / 2)[1])
transmissions = np.array(transmissions)

reference = transmissions[0] * np.exp(-2 * kappa * (widths - widths[0]))

plt.figure(figsize=(7, 4))
plt.semilogy(widths, transmissions, "o", color=IMAG, ms=7, label="simulation")
plt.semilogy(widths, reference, "--", color="#444",
             label=rf"$\propto e^{{-2\kappa l}},\ \kappa = {kappa:.0f}$")
plt.xlabel("barrier width l")
plt.ylabel("transmission T")
plt.title("Exponential decay of the transmission with barrier width")
plt.legend()
plt.show()

fitted_slope = np.polyfit(widths, np.log(transmissions), 1)[0]
print(f"fitted slope {fitted_slope:.0f}, to be compared with -2κ = {-2 * kappa:.0f}")

## 6. Further directions

The same tools can be applied to other configurations. A few examples:

- **Double well.** `sch.double_well(x, height, width, gap)` builds two barriers
  side by side, a standard model for the inversion of the ammonia molecule. A
  packet prepared in one well oscillates between the two.
- **Resonant transmission.** Varying the energy across a barrier reveals widths
  at which the transmission returns to one.
- **Harmonic trap.** Replacing the potential with $V(x) \propto (x - L/2)^2$
  produces a coherent state that oscillates back and forth in the well.

The double well is shown below as a starting point.

In [ ]:
V_double = sch.double_well(x, height=V0, width=0.02, gap=0.16)
psi = sch.evolve_crank_nicolson(psi0, V_double, dx, dt, n_steps)
spacetime(psi, "A packet in a double well")

---
*Numerical method after A. Goldberg, H. M. Schey and J. L. Swartz,
"Computer-Generated Motion Pictures of One-Dimensional Quantum-Mechanical
Transmission and Reflection Phenomena", American Journal of Physics 35, 177
(1967).*